In [1]:
import pandas as pd
import numpy as np
import os
 
os.makedirs("data/processed", exist_ok=True)

In [ ]:
#  Load raw data
print("=" * 60)
print("  Phase 2 — Data Preprocessing")
print("=" * 60)
 
print("Step 1: Loading raw data ")
 
crime_df = pd.read_csv("data/raw/crime_monthly_by_borough.csv")
socio_df = pd.read_csv("data/raw/socioeconomic_monthly.csv")
 
print(f"Crime data       :  {crime_df.shape[0]} rows × {crime_df.shape[1]} columns")
print(f"Socioeconomic    :  {socio_df.shape[0]} rows × {socio_df.shape[1]} columns")
 

  Phase 2 — Data Preprocessing

── Step 1: Loading raw data ─────────────────────────────
Crime data       :  1184 rows × 3 columns
Socioeconomic    :  1188 rows × 12 columns


In [ ]:
#  Merge on borough + month
print(" Step 2: Merging datasets ")
 
df = socio_df.merge(crime_df, on=["borough", "month"], how="left")
 
print(f"Merged shape     :  {df.shape[0]} rows × {df.shape[1]} columns")
 
missing_crime = df["crime_count"].isna().sum()
print(f"Missing crime    :  {missing_crime} rows")
if missing_crime > 0:
    print("  Missing rows:")
    for _, r in df[df["crime_count"].isna()].iterrows():
        print(f"    {r['borough']:<25} {r['month']}")


── Step 2: Merging datasets ─────────────────────────────
Merged shape     :  1188 rows × 13 columns
Missing crime    :  4 rows
  Missing rows:
    Camden                    2023-05
    Camden                    2023-07
    Camden                    2023-09
    Wandsworth                2023-10


In [ ]:
# Handle missing values
print(" Step 3: Handling missing values")
 
missing_before = df["crime_count"].isna().sum()
 
if missing_before > 0:
    print(f"Imputing {missing_before} missing values using borough-level interpolation...")
    df = df.sort_values(["borough", "month"]).reset_index(drop=True)
    df["crime_count"] = df.groupby("borough")["crime_count"].transform(
        lambda s: s.interpolate(method="linear", limit_direction="both")
    )
 
missing_after = df["crime_count"].isna().sum()
print(f"Missing before   :  {missing_before}")
print(f"Missing after    :  {missing_after}")
 
# Convert to int (interpolation produces floats)
df["crime_count"] = df["crime_count"].round().astype(int)
 


── Step 3: Handling missing values ──────────────────────
Imputing 4 missing values using borough-level interpolation...
Missing before   :  4
Missing after    :  0


In [ ]:
#  Feature engineering — temporal features
print("Step 4: Engineering temporal features ")
 
df["date"]        = pd.to_datetime(df["month"] + "-01")
df["year"]        = df["date"].dt.year
df["month_num"]   = df["date"].dt.month
df["quarter"]     = df["date"].dt.quarter
 
# Season (UK definitions)
season_map = {12: "winter", 1: "winter", 2: "winter",
               3: "spring", 4: "spring", 5: "spring",
               6: "summer", 7: "summer", 8: "summer",
               9: "autumn", 10: "autumn", 11: "autumn"}
df["season"] = df["month_num"].map(season_map)
 
# Cyclical encoding for month (preserves Dec→Jan continuity)
df["month_sin"] = np.sin(2 * np.pi * df["month_num"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month_num"] / 12)
 
print("Added: year, month_num, quarter, season, month_sin, month_cos")
 


── Step 4: Engineering temporal features ────────────────
Added: year, month_num, quarter, season, month_sin, month_cos


In [ ]:
#  Feature engineering — lag & rolling features
print("Step 5: Engineering lag & rolling features ")
 
# Sort properly before computing lag features
df = df.sort_values(["borough", "date"]).reset_index(drop=True)
 
# Lag features — crime in previous months (per borough)
df["crime_lag_1"]  = df.groupby("borough")["crime_count"].shift(1)
df["crime_lag_3"]  = df.groupby("borough")["crime_count"].shift(3)
df["crime_lag_12"] = df.groupby("borough")["crime_count"].shift(12)
 
# Rolling averages (using only past values via shift)
df["crime_rolling_3"]  = df.groupby("borough")["crime_count"].shift(1).rolling(3,  min_periods=1).mean().values
df["crime_rolling_6"]  = df.groupby("borough")["crime_count"].shift(1).rolling(6,  min_periods=1).mean().values
 
print("Added: crime_lag_1, crime_lag_3, crime_lag_12, crime_rolling_3, crime_rolling_6")
print("  Note: lag features use shift(1) to prevent data leakage")
 



── Step 5: Engineering lag & rolling features ───────────
Added: crime_lag_1, crime_lag_3, crime_lag_12, crime_rolling_3, crime_rolling_6
  Note: lag features use shift(1) to prevent data leakage


In [ ]:
# STEP 6: Calculate crime rate per 1,000 (secondary target)
print(" Step 6: Calculating crime rate per 1,000 population ")
 
df["crime_rate_per_1000"] = (df["crime_count"] / df["population_2023"]) * 1000
df["crime_rate_per_1000"] = df["crime_rate_per_1000"].round(2)
 
print(f"Average crime rate :  {df['crime_rate_per_1000'].mean():.2f} per 1,000 population")
print(f"Min                :  {df['crime_rate_per_1000'].min():.2f}  ({df.loc[df['crime_rate_per_1000'].idxmin(), 'borough']})")
print(f"Max                :  {df['crime_rate_per_1000'].max():.2f}  ({df.loc[df['crime_rate_per_1000'].idxmax(), 'borough']})")
 


── Step 6: Calculating crime rate per 1,000 population ──
Average crime rate :  12.99 per 1,000 population
Min                :  5.01  (Richmond upon Thames)
Max                :  115.11  (City of London)


In [ ]:
# STEP 7: Final cleanup and save
print("Step 7: Final cleanup ")
 
# Reordering columns logically
column_order = [
    # Identifiers
    "borough", "month", "date", "year", "month_num", "quarter", "season",
    "month_sin", "month_cos",
 
    # Targets
    "crime_count", "crime_rate_per_1000",
 
    # Lag/rolling features
    "crime_lag_1", "crime_lag_3", "crime_lag_12",
    "crime_rolling_3", "crime_rolling_6",
 
    # Socioeconomic features
    "imd_score", "income_deprivation_score", "employment_deprivation_score",
    "crime_deprivation_score",
    "population_2023", "population_density_per_km2",
    "claimant_count_rate_2023", "median_annual_earnings_2023",
    "median_house_price_2023", "overcrowding_rate",
]
 
df = df[column_order]
 
output_path = "data/processed/modelling_dataset.csv"
df.to_csv(output_path, index=False)
print(f"Saved            →  {output_path}")
print(f"Final shape      :  {df.shape[0]} rows × {df.shape[1]} columns")
 


── Step 7: Final cleanup ────────────────────────────────
Saved            →  data/processed/modelling_dataset.csv
Final shape      :  1188 rows × 26 columns


In [ ]:
#  Summary report
print(" Step 8: Generating preprocessing summary ")
 
summary = f"""
PREPROCESSING SUMMARY REPORT
============================
Input files:
  - crime_monthly_by_borough.csv   ({crime_df.shape[0]} rows)
  - socioeconomic_monthly.csv      ({socio_df.shape[0]} rows)
 
Output file:
  - modelling_dataset.csv          ({df.shape[0]} rows × {df.shape[1]} columns)
 
DECISIONS:
1. Merge strategy: LEFT JOIN socioeconomic ← crime
   Rationale: socioeconomic file has all 33 boroughs × 36 months = 1,188 expected rows.
   This ensures we detect any missing crime data.
 
2. Missing value imputation: {missing_before} missing crime counts
   Strategy: Linear interpolation within each borough's time series.
   Rationale: Missing values came from API connection errors during download,
   not real zero crime. Linear interpolation is appropriate for short gaps
   in temporal data.
 
3. Feature engineering:
   Temporal: year, month_num, quarter, season, month_sin, month_cos
   Lag:      crime_lag_1, crime_lag_3, crime_lag_12
   Rolling:  crime_rolling_3, crime_rolling_6
   All lag/rolling features use shift(1) to prevent data leakage.
 
4. Cyclical encoding: month_sin / month_cos
   Rationale: Tree-based models can capture non-linear month effects,
   but cyclical encoding preserves the December→January relationship
   for linear baseline models.
 
5. Secondary target: crime_rate_per_1000 = (crime_count / population) * 1000
   Allows fair comparison between boroughs of different sizes.
 
DATA RANGE:
  Months    : {df['month'].min()} to {df['month'].max()}  ({df['month'].nunique()} months)
  Boroughs  : {df['borough'].nunique()}
  Total obs : {len(df):,}
 
TARGET VARIABLE STATS:
  Mean crime count    : {df['crime_count'].mean():,.0f}
  Median              : {df['crime_count'].median():,.0f}
  Std                 : {df['crime_count'].std():,.0f}
  Min                 : {df['crime_count'].min():,}  ({df.loc[df['crime_count'].idxmin(), 'borough']}, {df.loc[df['crime_count'].idxmin(), 'month']})
  Max                 : {df['crime_count'].max():,}  ({df.loc[df['crime_count'].idxmax(), 'borough']}, {df.loc[df['crime_count'].idxmax(), 'month']})
 
ROWS WITH NaN (acceptable in lag features at series start):
"""
nan_counts = df.isna().sum()
for col, n in nan_counts.items():
    if n > 0:
        summary += f"  {col:<30} {n}\n"
summary += "\nNote: Lag features have NaN for early months — these will be handled in train/test split.\n"
 
with open("data/processed/preprocessing_summary.txt", "w") as f:
    f.write(summary)
 
print(summary)
 
print("\nPhase 2 complete ✓")
print("\nNext step: Phase 3 — Exploratory Data Analysis (EDA)")
 


── Step 8: Generating preprocessing summary ─────────────

PREPROCESSING SUMMARY REPORT
Input files:
  - crime_monthly_by_borough.csv   (1184 rows)
  - socioeconomic_monthly.csv      (1188 rows)

Output file:
  - modelling_dataset.csv          (1188 rows × 26 columns)

DECISIONS:
1. Merge strategy: LEFT JOIN socioeconomic ← crime
   Rationale: socioeconomic file has all 33 boroughs × 36 months = 1,188 expected rows.
   This ensures we detect any missing crime data.

2. Missing value imputation: 4 missing crime counts
   Strategy: Linear interpolation within each borough's time series.
   Rationale: Missing values came from API connection errors during download,
   not real zero crime. Linear interpolation is appropriate for short gaps
   in temporal data.

3. Feature engineering:
   Temporal: year, month_num, quarter, season, month_sin, month_cos
   Lag:      crime_lag_1, crime_lag_3, crime_lag_12
   Rolling:  crime_rolling_3, crime_rolling_6
   All lag/rolling features use shift(1) t